#### **OPTIMALIDAD DE LA SOLUCION - ALGORITMOS GREEDY**

In [41]:
import pandas as pd 
import matplotlib.pyplot as plt
from importlib import reload

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")
factibilidad = pd.read_csv("Factibilidad/factibilidad_5mm.csv")

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Calculamos también las cajas asignables a cada producto en la lista de productos.

In [42]:
def guardar_cajas_y_productos(grosor):
    caja_compras_merge = especificaciones_cajas.merge(procurement_cajas,
                                                  on="caja_tipo_id")

    cajas = [
        Caja(
            caja_id = row["caja_tipo_id"],
            dim_interior_ancho = row["caja_interior_ancho"],
            dim_interior_largo = row["caja_interior_largo"],
            dim_interior_alto = row["caja_interior_alto"],
            costo_unitario = row['costo_unitario_base']
        )
        for _, row in caja_compras_merge.iterrows()
    ]
    operaciones_planta_aux = operaciones_planta.drop('codigo_producto', axis=1) 
    
    prod_op_merge = pd.concat([catalogo_productos, operaciones_planta_aux], axis=1)

    productos = [
        Producto(
            codigo_producto = row['codigo_producto'],
            cantidad_paquetes = row['cantidad_paquetes'],
            peso_paquete = row['peso_neto_paquete'],
            demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
            demanda_curitiba = row['volumen_producto_planta_curitiba'],
            demanda_santiago = row['volumen_producto_planta_santiago'],
            demanda_monterrey = row['volumen_producto_planta_monterrey'],
            demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
            dim_producto_ancho = row['dim_producto_ancho'], 
            dim_producto_largo = row['dim_producto_largo'],
            dim_producto_alto = row['dim_producto_alto']
        )
        for _, row in prod_op_merge.iterrows()
    ]
    # Crear un diccionario de cajas por producto desde el CSV de factibilidad
    cajas_por_producto = {}
    for _, row in factibilidad.iterrows():
        codigo = row['codigo_producto']
        cajas_str = row['cajas_asignables_id']
        
        if isinstance(cajas_str, str) and cajas_str:
            # Separar por '; ' y limpiar
            cajas_list = [c.strip() for c in cajas_str.split(';') if c.strip()]
            cajas_por_producto[codigo] = cajas_list

    # Recorrer la lista de productos en orden y agregar las cajas
    for producto in productos:
        if producto.codigo_producto in cajas_por_producto:
            for caja_id in cajas_por_producto[producto.codigo_producto]:
                producto.agregar_caja_asignable(caja_id)
                
    #Elegir grosor
    for caja in cajas:
        caja.elegir_grosor(grosor_mm=grosor)
    return cajas, productos

#### **Greedy 0: Costos base sin descuentos**

Inicialmente, plantearemos una solución únicamente comparando los costos unitarios base, sin considerar todavía los descuentos por volúmenes anuales.

Como no hay ninguna restricción sobre la cantidad de cajas que podemos comprar de cada tipo, el problema se reduce en encontrar para cada producto el tipo de caja que más le convenga, es decir, el que ofrezca el menor costo.

Empecemos eligiendo un grosor de 5mm para todos los tipos de cajas, de manera que la restricción de headspace no supone ningún problema, pues las dimensiones internas de las cajas se diferencian hasta un 10% de las originales.

In [43]:
cajas, productos = guardar_cajas_y_productos(grosor=5)

In [44]:
def buscar_caja_por_id(id):
    caja_a_buscar = None
    for caja in cajas:
        if caja.caja_id == id:
            caja_a_buscar = caja
    return caja_a_buscar

In [45]:
solucion_base = Solucion(grosor=5)

for producto in productos:
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        asignacion = Asignacion(producto,caja)
        solucion_base.agregar_asignacion(asignacion, descuentos=False)

solucion_base.exportar_submmit(nombre_csv="1-base_5mm")
solucion_base.resumen_por_asignacion()

,codigo_producto,demanda_total,caja_id,utilizacion_pallet,utilizacion_caja,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,02cf77de65b70dd77905e2e33d78478f,0.323810,0.983137,927967.80,103109,15466350,16394317.80
1,BR0001,1546613,5e55b0f0e0122b55ba8d4d91fb921c5a,0.342702,0.927148,1005298.45,103109,15466350,16471648.45
2,BR0001,1546613,60eb77c934f99a121f410ec2037a2a46,0.339947,0.934944,927967.80,103109,15466350,16394317.80
3,BR0001,1546613,845990194eeb50cd131954a55fecd655,0.436688,0.972033,927967.80,77333,11599950,12527917.80
4,BR0001,1546613,8f26468860664ecfed551104179d7f3e,0.889973,0.952830,1005298.45,38667,5800050,6805348.45
...,...,...,...,...,...,...,...,...,...
5893,BR0421,145803,a7d151b83674a2f9c36356597a93821c,0.811290,0.997536,102062.10,2604,390600,492662.10
5894,BR0421,145803,d85b31778b741c8bdb462d9f332e56d2,0.834623,0.968591,94771.95,2604,390600,485371.95
5895,BR0421,145803,ea6403e25e3a52198dad19251c207a47,0.844404,0.956821,102062.10,2604,390600,492662.10
5896,BR0421,145803,ef2c4cbdd2be88c4d7a73fe0a429c842,0.817629,0.989366,102062.10,2604,390600,492662.10


#### **Greedy 1: Elección por menor costo**

Greedy (1)

Ordenamos los productos de menor a mayor según la cantidad de tipos de cajas asignables.
Iteramos los productos p desde el menor:
    (HACER EN FACTIBILIDAD) Ver factibilidad de caja por resistencia a comprension + headspace
    Asignar a p el tipo de caja con menor costo considerando el cálculo de descuentos sumando los volúmenes de producto
    Recalculo descuentos

Con los descuentos finales, vuelvo a calcular cada costo de producto

Guardamos nuevamente cajas y productos, reiniciando toda asignación previa:

In [46]:
cajas, productos = guardar_cajas_y_productos(grosor=5)

Ordenamos los productos según la cantidad de cajas asignables de menor a mayor

In [47]:
def ordenar_segun_cantidad_cajas_asignables(productos):
    productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))
    return productos_ordenados

In [48]:
productos_ordenados = ordenar_segun_cantidad_cajas_asignables(productos)

In [49]:
solucion_greedy1 = Solucion(grosor=5)

for producto in productos_ordenados:
    costos_totales = {}
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        asignacion = Asignacion(producto, caja)
        caja.asignar_producto(producto)
        costo_packaging = asignacion.costo_packaging_producto_total()
        costo_flete = 150 * asignacion.cant_pallets_requeridas()
        
        costo_total = costo_packaging + costo_flete
        costos_totales[caja_id] = costo_total
        
        caja.revocar_producto(producto)
    
    caja_id_optima = min(costos_totales, key=costos_totales.get)
    caja_optima = buscar_caja_por_id(caja_id_optima)
    
    asignacion = Asignacion(producto, caja_optima)
    solucion_greedy1.agregar_asignacion(asignacion)
    
solucion_greedy1.exportar_submmit(nombre_csv="-greedy1_5mm")
solucion_greedy1.resumen_por_asignacion()

,codigo_producto,demanda_total,caja_id,utilizacion_pallet,utilizacion_caja,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,8f26468860664ecfed551104179d7f3e,0.889973,0.952830,715185.835,38667,5800050,6515235.835
1,BR0002,139211,0835ff365412a67b720a19713ec250f3,0.937728,0.990522,72389.720,2901,435150,507539.720
2,BR0003,172506,4f08f853cf7f9be28571df66a50bc322,0.955421,1.000000,89703.120,2157,323550,413253.120
3,BR0004,271715,5914abeae491eab8612131985b15994f,0.945868,0.992626,123630.325,4853,727950,851580.325
4,BR0005,7586,0835ff365412a67b720a19713ec250f3,0.937728,0.910072,3944.720,159,23850,27794.720
...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,1416893384452893b004453859dc1ae4,0.818193,0.979079,1256.255,50,7500,8756.255
423,BR0418,3942,1416893384452893b004453859dc1ae4,0.818193,0.979079,1793.610,71,10650,12443.610
424,BR0419,43300,161332db17a4c8bcba8a5651098cbe91,0.983425,0.957743,22516.000,602,90300,112816.000
425,BR0420,205852,1cbc364190e4feb6cfbe064584c400ab,0.768904,0.960870,95237.415,3677,551550,646787.415


#### **Greedy 2: Elección por mayor volumen**

Greedy (2)

Ordenamos los productos de menor a mayor según la cantidad de tipos de cajas asignables.
Iteramos los productos p desde el menor:
    (HACER EN FACTIBILIDAD) Ver factibilidad de caja por resistencia a comprension + headspace
    Asignar a p el tipo de caja con mayor volúmen actualmente

Con los descuentos finales, vuelvo a calcular cada costo de producto

In [50]:
cajas, productos = guardar_cajas_y_productos(grosor=5)
productos_ordenados = ordenar_segun_cantidad_cajas_asignables(productos)

In [51]:
solucion_greedy2 = Solucion(grosor=5)

for producto in productos_ordenados:
    metricas = {}  # caja_id -> (volumen_total, costo_total)
    
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        
        # volumen requerido actual (antes de asignar este producto)
        volumen_total = caja.unidades_total_requeridas()
        
        # costo simulado de asignar este producto a esta caja
        asignacion = Asignacion(producto, caja)
        caja.asignar_producto(producto)
        costo_packaging = asignacion.costo_packaging_producto_total()
        costo_flete = 150 * asignacion.cant_pallets_requeridas()
        costo_total = costo_packaging + costo_flete
        caja.revocar_producto(producto)
        
        metricas[caja_id] = (volumen_total, costo_total)
    
    # criterio: mayor volumen_total primero, y en caso de empate, menor costo_total
    caja_id_optima = max(metricas, key=lambda id: (metricas[id][0], -metricas[id][1]))
    caja_optima = buscar_caja_por_id(caja_id_optima)
    
    asignacion = Asignacion(producto, caja_optima)
    solucion_greedy2.agregar_asignacion(asignacion)

solucion_greedy2.exportar_submmit(nombre_csv="-greedy2_5mm")
solucion_greedy2.resumen_por_asignacion()

,codigo_producto,demanda_total,caja_id,utilizacion_pallet,utilizacion_caja,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,5e55b0f0e0122b55ba8d4d91fb921c5a,0.342702,0.927148,715185.835,103109,15466350,1.618154e+07
1,BR0002,139211,0835ff365412a67b720a19713ec250f3,0.937728,0.990522,72389.720,2901,435150,5.075397e+05
2,BR0003,172506,67894c5a8355ca9b7e57b96362fa45f1,0.982876,0.970414,89703.120,2157,323550,4.132531e+05
3,BR0004,271715,5914abeae491eab8612131985b15994f,0.945868,0.992626,123630.325,4853,727950,8.515803e+05
4,BR0005,7586,0835ff365412a67b720a19713ec250f3,0.937728,0.910072,3944.720,159,23850,2.779472e+04
...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,1416893384452893b004453859dc1ae4,0.818193,0.979079,1256.255,50,7500,8.756255e+03
423,BR0418,3942,1416893384452893b004453859dc1ae4,0.818193,0.979079,1793.610,71,10650,1.244361e+04
424,BR0419,43300,161332db17a4c8bcba8a5651098cbe91,0.983425,0.957743,22516.000,602,90300,1.128160e+05
425,BR0420,205852,1cbc364190e4feb6cfbe064584c400ab,0.768904,0.960870,95237.415,3677,551550,6.467874e+05


#### **Greedy 3: Ordenamiento de productos según su demanda total**

Greedy (3)

Rehacer Greedy 1 y 2 ordenando los productos de mayor a menor según la demanda.

In [52]:
def ordenar_segun_demanda(productos):
    productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)
    return productos_ordenados

In [53]:
cajas, productos = guardar_cajas_y_productos(grosor=5)

productos_ordenados = ordenar_segun_demanda(productos)

In [54]:
solucion_greedy3 = Solucion(grosor=5)

for producto in productos_ordenados:
    costos_totales = {}
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        asignacion = Asignacion(producto, caja)
        caja.asignar_producto(producto)
        costo_packaging = asignacion.costo_packaging_producto_total()
        costo_flete = 150 * asignacion.cant_pallets_requeridas()
        
        costo_total = costo_packaging + costo_flete
        costos_totales[caja_id] = costo_total
        
        caja.revocar_producto(producto)
    
    caja_id_optima = min(costos_totales, key=costos_totales.get)
    caja_optima = buscar_caja_por_id(caja_id_optima)
    
    asignacion = Asignacion(producto, caja_optima)
    solucion_greedy3.agregar_asignacion(asignacion)
    
solucion_greedy3.exportar_submmit(nombre_csv="-greedy3_5mm")
solucion_greedy3.resumen_por_asignacion()

,codigo_producto,demanda_total,caja_id,utilizacion_pallet,utilizacion_caja,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,8f26468860664ecfed551104179d7f3e,0.889973,0.952830,715185.835,38667,5800050,6515235.835
1,BR0002,139211,0835ff365412a67b720a19713ec250f3,0.937728,0.990522,72389.720,2901,435150,507539.720
2,BR0003,172506,4f08f853cf7f9be28571df66a50bc322,0.955421,1.000000,89703.120,2157,323550,413253.120
3,BR0004,271715,5914abeae491eab8612131985b15994f,0.945868,0.992626,123630.325,4853,727950,851580.325
4,BR0005,7586,3249c6671a2e400c91ae940376aa4946,0.915886,0.932642,3451.630,159,23850,27301.630
...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,1416893384452893b004453859dc1ae4,0.818193,0.979079,1256.255,50,7500,8756.255
423,BR0418,3942,1416893384452893b004453859dc1ae4,0.818193,0.979079,1793.610,71,10650,12443.610
424,BR0419,43300,161332db17a4c8bcba8a5651098cbe91,0.983425,0.957743,22516.000,602,90300,112816.000
425,BR0420,205852,1cbc364190e4feb6cfbe064584c400ab,0.768904,0.960870,95237.415,3677,551550,646787.415


In [55]:
cajas, productos = guardar_cajas_y_productos(grosor=5)

productos_ordenados = ordenar_segun_demanda(productos)

In [56]:
solucion_greedy4 = Solucion(grosor=5)

for producto in productos_ordenados:
    metricas = {}  # caja_id -> (volumen_total, costo_total)
    
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        
        # volumen requerido actual (antes de asignar este producto)
        volumen_total = caja.unidades_total_requeridas()
        
        # costo simulado de asignar este producto a esta caja
        asignacion = Asignacion(producto, caja)
        caja.asignar_producto(producto)
        costo_packaging = asignacion.costo_packaging_producto_total()
        costo_flete = 150 * asignacion.cant_pallets_requeridas()
        costo_total = costo_packaging + costo_flete
        caja.revocar_producto(producto)
        
        metricas[caja_id] = (volumen_total, costo_total)
    
    # criterio: mayor volumen_total primero, y en caso de empate, menor costo_total
    caja_id_optima = max(metricas, key=lambda id: (metricas[id][0], -metricas[id][1]))
    caja_optima = buscar_caja_por_id(caja_id_optima)
    
    asignacion = Asignacion(producto, caja_optima)
    solucion_greedy4.agregar_asignacion(asignacion)

solucion_greedy4.exportar_submmit(nombre_csv="-greedy4_5mm")
solucion_greedy4.resumen_por_asignacion()

,codigo_producto,demanda_total,caja_id,utilizacion_pallet,utilizacion_caja,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,8f26468860664ecfed551104179d7f3e,0.889973,0.952830,715185.835,38667,5800050,6515235.835
1,BR0002,139211,0835ff365412a67b720a19713ec250f3,0.937728,0.990522,72389.720,2901,435150,507539.720
2,BR0003,172506,4f08f853cf7f9be28571df66a50bc322,0.955421,1.000000,89703.120,2157,323550,413253.120
3,BR0004,271715,5914abeae491eab8612131985b15994f,0.945868,0.992626,123630.325,4853,727950,851580.325
4,BR0005,7586,0835ff365412a67b720a19713ec250f3,0.937728,0.910072,3944.720,159,23850,27794.720
...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,1416893384452893b004453859dc1ae4,0.818193,0.979079,1256.255,50,7500,8756.255
423,BR0418,3942,1416893384452893b004453859dc1ae4,0.818193,0.979079,1793.610,71,10650,12443.610
424,BR0419,43300,161332db17a4c8bcba8a5651098cbe91,0.983425,0.957743,22516.000,602,90300,112816.000
425,BR0420,205852,1cbc364190e4feb6cfbe064584c400ab,0.768904,0.960870,95237.415,3677,551550,646787.415


Partiendo de la solucion, veo si la puedo mejorar

En cada paso:
    Cambio el tipo de caja, y me fijo si con los descuentos puedo tener un menor costo

Restriccion: dim_actual <= dim_producto x 1.1

Headspace (grosor 4.5): dim_actual * 0.9 >= dim_producto

dim_actual >= dim_producto : 0.9
dim_actual <= dim_producto x 1.1

dim_producto x 1.11 <= dim_actual <= dim_producto x 1.1